# .mat 파일 내용 확인
- Ray Tracing 결과로 저장된 multipath 파라미터 시각화
- `scenarios/DT31/wireless/scene_{N}/BS1_UE_0-1.mat` 기준

| Row | 파라미터 | 단위 |
|-----|----------|------|
| 0 | Phase | deg |
| 1 | ToA (delay) | s |
| 2 | Rx Power | dBm |
| 3 | DoA azimuth (phi) | deg |
| 4 | DoA elevation (theta) | deg |
| 5 | DoD azimuth (phi) | deg |
| 6 | DoD elevation (theta) | deg |
| 7 | LoS flag | 0/1 |
| 8 | Doppler velocity | m/s |
| 9 | Doppler acceleration | m/s² |

In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── 파일 경로 설정 ─────────────────────────────────────────────
SCENE_IDX = 0
MAT_PATH  = f"scenarios/DT31/wireless/scene_{SCENE_IDX}/BS1_UE_0-1.mat"

mat = sio.loadmat(MAT_PATH)
p   = mat["channels"][0, 0]["p"][0, 0]   # (10, P)
P   = p.shape[1]

# Row 의미 매핑 (RayTracingLoader.py 기준)
ROW_NAMES = {
    0: ("Phase",             "deg"),
    1: ("ToA (delay)",       "s"),
    2: ("Rx Power",          "dBm"),
    3: ("DoA azimuth",       "deg"),
    4: ("DoA elevation",     "deg"),
    5: ("DoD azimuth",       "deg"),
    6: ("DoD elevation",     "deg"),
    7: ("LoS flag",          "0/1"),
    8: ("Doppler vel",       "m/s"),
    9: ("Doppler acc",       "m/s²"),
}

# ── 기본 정보 출력 ─────────────────────────────────────────────
print("=" * 55)
print(f"  Scene {SCENE_IDX}  |  {MAT_PATH}")
print("=" * 55)

tx = mat["tx_loc"].squeeze()
rx = mat["rx_locs"].squeeze()
print(f"  BS 위치  (x, y, z) : {tx[0]:.2f}, {tx[1]:.2f}, {tx[2]:.2f}")
print(f"  UE 위치  (x, y, z) : {rx[0]:.2f}, {rx[1]:.2f}, {rx[2]:.2f}")
print(f"  BS-UE 거리          : {rx[3]:.2f} m")
print(f"  Path Loss           : {rx[4]:.2f} dB")
print(f"  총 multipath 수     : {P}")
print(f"  LoS 경로 수         : {int(p[7].sum())}")
print()
print(f"  {chr(34)}Row{chr(34):<4} {chr(34)}파라미터{chr(34):<20} {chr(34)}단위{chr(34):<8} {chr(34)}min{chr(34):>12} {chr(34)}max{chr(34):>12} {chr(34)}mean{chr(34):>12}")
print("  " + "-" * 72)
for i in range(10):
    name, unit = ROW_NAMES[i]
    v = p[i]
    print(f"  {i:<5} {name:<22} {unit:<8} {v.min():>12.4g} {v.max():>12.4g} {v.mean():>12.4g}")


## 각 Row 상위 5개 값 확인

In [ ]:
print(f"총 경로 수: {P}개\n")
print(f"{'Row':<5} {'파라미터':<22} {'단위':<8}  상위 5개 값")
print("-" * 100)
for i in range(10):
    name, unit = ROW_NAMES[i]
    v = p[i]
    if i == 7:  # LoS flag는 앞 5개
        top5 = v[:5]
        label = "(first 5)"
    else:
        top5 = np.sort(v)[::-1][:5]
        label = "(내림차순)"
    vals_str = "  ".join(f"{x:>13.5g}" for x in top5)
    print(f"{i:<5} {name:<22} {unit:<8}  {vals_str}  {label}")


## 히스토그램 분포 시각화

In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle(f"Scene {SCENE_IDX} — Ray Tracing Multipath Parameters (N={P})", fontsize=13, fontweight="bold")
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)
colors = plt.cm.tab10.colors

def hist_ax(ax, row, bins=40, color="steelblue"):
    name, unit = ROW_NAMES[row]
    ax.hist(p[row], bins=bins, color=color, edgecolor="white", linewidth=0.3)
    ax.set_title(f"Row {row}: {name}", fontsize=10)
    ax.set_xlabel(unit, fontsize=8)
    ax.set_ylabel("# paths", fontsize=8)
    ax.tick_params(labelsize=7)

# Row 2: Rx Power
hist_ax(fig.add_subplot(gs[0, 0]), 2, color=colors[0])

# Row 1: ToA (x축 단위 변환)
ax = fig.add_subplot(gs[0, 1])
hist_ax(ax, 1, color=colors[1])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x*1e7:.1f}"))
ax.set_xlabel("s (×1e-7)", fontsize=8)

# Row 0: Phase
hist_ax(fig.add_subplot(gs[0, 2]), 0, color=colors[2])

# Row 3: DoA azimuth
hist_ax(fig.add_subplot(gs[1, 0]), 3, color=colors[3])

# Row 4: DoA elevation
hist_ax(fig.add_subplot(gs[1, 1]), 4, color=colors[4])

# Row 5: DoD azimuth
hist_ax(fig.add_subplot(gs[1, 2]), 5, color=colors[5])

# Row 6: DoD elevation
hist_ax(fig.add_subplot(gs[2, 0]), 6, color=colors[6])

# Row 8: Doppler velocity
hist_ax(fig.add_subplot(gs[2, 1]), 8, color=colors[7])

# Row 7: LoS flag (bar chart)
ax = fig.add_subplot(gs[2, 2])
los_counts = [int((p[7] == 0).sum()), int((p[7] == 1).sum())]
ax.bar(["NLoS", "LoS"], los_counts, color=[colors[8], colors[9]])
ax.set_title("Row 7: LoS flag", fontsize=10)
ax.set_ylabel("# paths", fontsize=8)
ax.tick_params(labelsize=7)
for k, v in enumerate(los_counts):
    ax.text(k, v + 2, str(v), ha="center", fontsize=9, fontweight="bold")

plt.savefig(f"scene{SCENE_IDX}_multipath_hist.png", dpi=120, bbox_inches="tight")
plt.show()


## Power Delay Profile (PDP) — ToA vs Rx Power

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

toa   = p[1]          # Row 1: ToA (s)
power = p[2]          # Row 2: Rx Power (dBm)
los   = p[7].astype(bool)

# NLOS 먼저, LOS 위에 덮어씌움
ax.scatter(toa[~los] * 1e9, power[~los], s=6, alpha=0.5, color="steelblue", label="NLoS")
ax.scatter(toa[los]  * 1e9, power[los],  s=40, marker="*", color="red",       label="LoS")

ax.set_xlabel("ToA (ns)", fontsize=11)
ax.set_ylabel("Rx Power (dBm)", fontsize=11)
ax.set_title(f"Scene {SCENE_IDX} — Power Delay Profile (N={P} paths)", fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"scene{SCENE_IDX}_pdp.png", dpi=120, bbox_inches="tight")
plt.show()


## DoA / DoD 산점도 (Azimuth vs Elevation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Scene {SCENE_IDX} — Angle Distribution", fontsize=13, fontweight="bold")

los = p[7].astype(bool)

for ax, (phi_row, theta_row, title) in zip(axes, [
    (3, 4, "DoA  (Angle of Arrival)"),
    (5, 6, "DoD  (Angle of Departure)"),
]):
    phi   = p[phi_row]
    theta = p[theta_row]
    sc = ax.scatter(phi[~los], theta[~los], s=5, alpha=0.4, c="steelblue", label="NLoS")
    ax.scatter(phi[los], theta[los], s=60, marker="*", c="red", label="LoS", zorder=5)
    ax.set_xlabel("Azimuth / phi (deg)", fontsize=10)
    ax.set_ylabel("Elevation / theta (deg)", fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"scene{SCENE_IDX}_angles.png", dpi=120, bbox_inches="tight")
plt.show()


## Doppler velocity 분포 (LoS vs NLoS)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

dop = p[8]   # Row 8: Doppler velocity
los = p[7].astype(bool)

ax.hist(dop[~los], bins=50, alpha=0.7, color="steelblue", label=f"NLoS ({(~los).sum()}개)")
ax.hist(dop[los],  bins=10, alpha=0.9, color="red",       label=f"LoS  ({los.sum()}개)")
ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("Doppler velocity (m/s)", fontsize=11)
ax.set_ylabel("# paths", fontsize=11)
ax.set_title(f"Scene {SCENE_IDX} — Doppler Velocity Distribution", fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"scene{SCENE_IDX}_doppler.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"LoS  Doppler vel : {dop[los].mean():.4f} m/s (mean)")
print(f"NLoS Doppler vel : {dop[~los].mean():.4f} m/s (mean)")
